In [29]:

import random
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Any
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score, auc

# Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Dataviz
from matplotlib.colors import rgb2hex
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import matplotlib
import seaborn as sns
import plotly.express as px


In [35]:

# --- Dataset Creation ---
X, y = make_classification(n_samples=1000, n_features=10, n_informative=4, flip_y=0.2, weights=[0.8, 0.1],random_state=42)
y = y.reshape(-1, 1)
df = pd.DataFrame(np.concatenate([X, y], axis=1))

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=None, train_size=None, random_state=None, shuffle=True, stratify=None)
df_train, df_test = train_test_split(df, test_size=None, train_size=None, random_state=None, shuffle=True, stratify=None)

x_columns = df.columns[:10]
y_column = df.columns[10]

X_train_df = df_train[x_columns]
X_test_df = df_test[x_columns]
y_train_df = df_train[y_column]
y_test_df = df_test[y_column]

model = LogisticRegression()
model.fit(X_train_df, y_train_df)

LogisticRegression()

In [ ]:

def plotly_roc_auc(dataset_dict):

    fig = go.Figure()
    fig.add_shape(
        type='line', line=dict(dash='dash'),
        x0=0, x1=1, y0=0, y1=1
    )
    for set_name in dataset_dict:
        y_ = dataset_dict[set_name]['y']
        y_prob = dataset_dict[set_name]['y_prob'][:, 1]

        fpr, tpr, thresholds = roc_curve(y_, y_prob)
        auc_score = auc(fpr, tpr)
        fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f"{set_name} (AUC={auc_score:.3f})", mode='lines'))

    fig.update_layout(
        title=f"ROC AUC Curve",
        title_x=0.5,
        xaxis=dict(
            title=dict(text='False Positive Rate'),
            constrain='domain'
        ),
        yaxis=dict(
            title=dict(text='True Positive Rate'),
            scaleanchor='x',
            scaleratio=1
        ),
        width=700, height=500
    )
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.update_xaxes(constrain='domain')
    return fig


dataset_dict = {
    "Train":
    {
        "x": X_train_df,
        "y": y_train_df,
        "y_pred": model.predict(X_train_df),
        "y_prob": model.predict_proba(X_train_df),
    },
    "Test":
    {
        "x": X_test_df,
        "y": y_test_df,
        "y_pred": model.predict(X_test_df),
        "y_prob": model.predict_proba(X_test_df),
    },
}

plotly_roc_auc(dataset_dict)
